In [1]:
# -------------------------------
# 0. LOOKUP Dictionary
# -------------------------------
SCENARIO_LOOKUP = {
    # Natural Faults (1–6)
    1: "Fault L1 (10–19%)",
    2: "Fault L1 (20–79%)",
    3: "Fault L1 (80–90%)",
    4: "Fault L2 (10–19%)",
    5: "Fault L2 (20–79%)",
    6: "Fault L2 (80–90%)",

    # Maintenance (13–14)
    13: "Line Maintenance L1",
    14: "Line Maintenance L2",

    # Data Injection Attacks (7–12)
    7:  "Data Injection: L1 Fault 10–19% with tripping",
    8:  "Data Injection: L1 Fault 20–79% with tripping",
    9:  "Data Injection: L1 Fault 80–90% with tripping",
    10: "Data Injection: L2 Fault 10–19% with tripping",
    11: "Data Injection: L2 Fault 20–79% with tripping",
    12: "Data Injection: L2 Fault 80–90% with tripping",

    # Remote Tripping Attacks (15–20)
    15: "Remote Tripping: Command Injection R1",
    16: "Remote Tripping: Command Injection R2",
    17: "Remote Tripping: Command Injection R3",
    18: "Remote Tripping: Command Injection R4",
    19: "Remote Tripping: Command Injection R1 & R2",
    20: "Remote Tripping: Command Injection R3 & R4",

    # Relay Setting Change (21–30)
    21: "Relay Setting Change: R1 disabled (L1 10–19% fault)",
    22: "Relay Setting Change: R1 disabled (L1 20–90% fault)",
    23: "Relay Setting Change: R2 disabled (L1 10–49% fault)",
    24: "Relay Setting Change: R2 disabled (L1 50–79% fault)",
    25: "Relay Setting Change: R2 disabled (L1 80–90% fault)",
    26: "Relay Setting Change: R3 disabled (L2 10–19% fault)",
    27: "Relay Setting Change: R3 disabled (L2 20–49% fault)",
    28: "Relay Setting Change: R3 disabled (L2 50–90% fault)",
    29: "Relay Setting Change: R4 disabled (L2 10–79% fault)",
    30: "Relay Setting Change: R4 disabled (L2 80–90% fault)",

    # Relay Setting Change (two relays + fault)
    35: "Relay Setting Change: R1 & R2 disabled (L1 10–49% fault)",
    36: "Relay Setting Change: R1 & R2 disabled (L1 50–90% fault)",
    37: "Relay Setting Change: R3 & R4 disabled (L1 10–49% fault)",
    38: "Relay Setting Change: R3 & R4 disabled (L1 50–90% fault)",

    # Relay Setting Change (two relays + maintenance)
    39: "Relay Setting Change: R1 & R2 disabled during maintenance",
    40: "Relay Setting Change: R1 & R2 disabled during maintenance",

    # Normal Operation
    41: "Normal Operation (no disturbances)"
}

DI_NAMES = {
    7: "DI: Fault 10–19% L1",
    8: "DI: Fault 20–79% L1",
    9: "DI: Fault 80–90% L1",
    10: "DI: Fault 10–19% L2",
    11: "DI: Fault 20–79% L2",
    12: "DI: Fault 80–90% L2"
}

RT_NAMES = {
    15: "Remote Trip: Cmd R1",
    16: "Remote Trip: Cmd R2",
    17: "Remote Trip: Cmd R3",
    18: "Remote Trip: Cmd R4",
    19: "Remote Trip: Cmd R1 & R2",
    20: "Remote Trip: Cmd R3 & R4"
}

RSC_NAMES = {
    21: "RSC: L1 R1 Disabled (10–19%)",
    22: "RSC: L1 R1 Disabled (20–90%)",
    23: "RSC: L1 R2 Disabled (10–49%)", 
    24: "RSC: L1 R2 Disabled (50–79%)",
    25: "RSC: L1 R2 Disabled (80–90%)",
    26: "RSC: L2 R3 Disabled (10–19%)",
    27: "RSC: L2 R3 Disabled (20–49%)",
    28: "RSC: L2 R3 Disabled (50–90%)",
    29: "RSC: L2 R4 Disabled (10–79%)",
    30: "RSC: L2 R4 Disabled (80–90%)",
    35: "RSC: L1 R1&R2 Disabled (10–49%)",
    36: "RSC: L1 R1&R2 Disabled (50–90%)",
    37: "RSC: L1 R3&R4 Disabled (10–49%)",
    38: "RSC: L1 R3&R4 Disabled (50–90%)",
    39: "RSC: L1 Maint R1&R2 Disabled",
    40: "RSC: L1 Maint R1&R2 Disabled"
}

DATASETS = {
    "M1": "M1_Attack_vs_NonAttack.csv",
    "M2": "M2_Natural_3Class.csv",
    "M3": "M3_Attack_3Class.csv",
    "M4": "M4_DataInjection_Internal.csv",
    "M5": "M5_RemoteTripping_Internal.csv",
    "M6": "M6_RelaySetting_Internal.csv"
}

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

In [5]:
import pandas as pd
import numpy as np
import os

DATA_DIR = "../data/datasets_hierarchical"

datasets = {
    "M1_Attack_vs_NonAttack": "M1_Attack_vs_NonAttack.csv",
    "M2_Natural_3Class": "M2_Natural_3Class.csv",
    "M3_Attack_3Class": "M3_Attack_3Class.csv",
    "M4_DataInjection_Internal": "M4_DataInjection_Internal.csv",
    "M5_RemoteTripping_Internal": "M5_RemoteTripping_Internal.csv",
    "M6_RelaySetting_Internal": "M6_RelaySetting_Internal.csv",
}

cleaned_datasets = {}

print("\n==================== CLEANING RESULTS ====================\n")

for name, filename in datasets.items():

    path = os.path.join(DATA_DIR, filename)
    df = pd.read_csv(path)

    # BEFORE
    original_shape = df.shape

    # Convert Inf → NaN
    df = df.replace([np.inf, -np.inf], np.nan)

    # Fill NaN column-wise median
    df = df.fillna(df.median(numeric_only=True))

    # AFTER
    cleaned_shape = df.shape

    cleaned_datasets[name] = df

    print(f"{name}:")
    print(f"   Before cleaning: {original_shape[0]} rows × {original_shape[1]} columns")
    print(f"   After cleaning:  {cleaned_shape[0]} rows × {cleaned_shape[1]} columns")
    print("   ✔ Cleaned: Inf→NaN→Median filled\n")

print("\n===========================================================\n")


==================== CLEANING RESULTS ====================

M1_Attack_vs_NonAttack:
   Before cleaning: 78377 rows × 131 columns
   After cleaning:  78377 rows × 131 columns
   ✔ Cleaned: Inf→NaN→Median filled

M2_Natural_3Class:
   Before cleaning: 22714 rows × 131 columns
   After cleaning:  22714 rows × 131 columns
   ✔ Cleaned: Inf→NaN→Median filled

M3_Attack_3Class:
   Before cleaning: 55663 rows × 131 columns
   After cleaning:  55663 rows × 131 columns
   ✔ Cleaned: Inf→NaN→Median filled

M4_DataInjection_Internal:
   Before cleaning: 9582 rows × 130 columns
   After cleaning:  9582 rows × 130 columns
   ✔ Cleaned: Inf→NaN→Median filled

M5_RemoteTripping_Internal:
   Before cleaning: 8737 rows × 130 columns
   After cleaning:  8737 rows × 130 columns
   ✔ Cleaned: Inf→NaN→Median filled

M6_RelaySetting_Internal:
   Before cleaning: 37344 rows × 130 columns
   After cleaning:  37344 rows × 130 columns
   ✔ Cleaned: Inf→NaN→Median filled





In [ ]:
def load_dataset(path, feature_cols, target_col):
    df = pd.read_csv(path)
    df = clean_dataset(df)
    X = df[feature_cols].copy()
    y = df[target_col].copy()
    return X, y

In [ ]:
def stratified_70_15_15_split(X, y, random_state=42):
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=random_state
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=random_state
    )

    return X_train, X_val, X_test, y_train, y_val, y_test


In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test, class_names):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    print(f"\n===== {name} =====")

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average='macro', zero_division=0)
    rec = recall_score(y_test, preds, average='macro', zero_division=0)
    f1_macro = f1_score(y_test, preds, average='macro', zero_division=0)

    print("Accuracy:", acc)
    print("Precision:", prec)
    print("Recall:", rec)
    print("Macro F1:", f1_macro)

    labels_present = np.unique(y_test)
    names_present = [str(class_names[i]) for i in labels_present]

    print("\nClassification Report:")
    print(classification_report(
        y_test, preds, labels=labels_present,
        target_names=names_present, zero_division=0
    ))

    cm = confusion_matrix(y_test, preds, labels=labels_present)

    plt.figure(figsize=(6,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=names_present, yticklabels=names_present)
    plt.title(f"{name} - Confusion Matrix")
    plt.tight_layout()
    plt.show()

    return {
        "Model": name,
        "Accuracy": acc,
        "Precision_macro": prec,
        "Recall_macro": rec,
        "F1_macro": f1_macro
    }
